# pet segmentation from ascii images

This notebook contains my baseline for the NEOAI 2026 Terminal Cuties Segmentation competition.

Task: binary image segmentation  
Public Dice: **0.3048**  
Result: **68th overall and around top 2-3 in the team selection**  
Training data: competition data only

### a small clarification

The competition provided the task, data, metric, statistical context, and a starter baseline. I wrote and adapted the Kaggle code myself, including the preprocessing, compact segmentation model, training loop, validation, inference, and submission pipeline.

I am currently studying statistics independently, so the terminology here reflects what the problem requires, not a claim that I already know everything.

The model weights start from random initialization, without external data or pretrained weights.


## approach

The original images are 1280 by 1280 pixels, but every ASCII character occupies a fixed 10 by 10 block. I resize each image and mask to a 128 by 128 character grid.

The model is a lightweight fully convolutional encoder-decoder. I kept it small cuz I wanted a clear baseline that was easy to test and improve. It is tiny, but at least it trains fast. hohoho.

Training uses `BCEWithLogitsLoss`, Adam, and four epochs. During inference, predictions are thresholded at 0.5, resized to the original resolution, and encoded with column-major run-length encoding.


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BASE_DIR = Path('/kaggle/input/competitions/neoai-2026-day-1-cv')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## data preparation

RGB values are normalized to the 0 to 1 range. Masks use nearest-neighbor resizing to preserve binary labels. Random horizontal flips provide basic augmentation.


In [ ]:
train_df = pd.read_csv(BASE_DIR / 'train.csv')
test_df = pd.read_csv(BASE_DIR / 'test.csv')
train_part, valid_part = train_test_split(
    train_df, test_size=0.2, random_state=SEED
)
print(f'Train: {len(train_part)} | Validation: {len(valid_part)} | Test: {len(test_df)}')

In [ ]:
class PetGridDataset(Dataset):
    def __init__(self, frame, base_dir, training=False, with_masks=True):
        self.frame = frame.reset_index(drop=True)
        self.base_dir = base_dir
        self.training = training
        self.with_masks = with_masks

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(self.base_dir / row.img).convert('RGB')
        image = image.resize((128, 128), Image.Resampling.BOX)
        image = np.asarray(image, dtype=np.float32) / 255.0

        if not self.with_masks:
            return torch.from_numpy(image.transpose(2, 0, 1))

        mask = Image.open(self.base_dir / row.label).convert('L')
        mask = mask.resize((128, 128), Image.Resampling.NEAREST)
        mask = (np.asarray(mask) > 0).astype(np.float32)

        if self.training and random.random() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])

        image = torch.from_numpy(image.transpose(2, 0, 1))
        mask = torch.from_numpy(mask).unsqueeze(0)
        return image, mask

## model

The network has three encoder stages and a compact decoder. It does not use skip connections, so some fine boundary detail is lost. This keeps the architecture simple but limits segmentation quality.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class PixelPawsLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(3, 16), nn.MaxPool2d(2),
            ConvBlock(16, 32), nn.MaxPool2d(2),
            ConvBlock(32, 64), nn.MaxPool2d(2),
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(64, 32),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(32, 16),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(16, 1, kernel_size=1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
train_loader = DataLoader(
    PetGridDataset(train_part, BASE_DIR, training=True),
    batch_size=16, shuffle=True, num_workers=2, pin_memory=True
)
valid_loader = DataLoader(
    PetGridDataset(valid_part, BASE_DIR),
    batch_size=16, shuffle=False, num_workers=2, pin_memory=True
)

model = PixelPawsLite().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def dice_score(logits, targets, threshold=0.5, eps=1e-7):
    predictions = (torch.sigmoid(logits) > threshold).float()
    intersection = (predictions * targets).sum(dim=(1, 2, 3))
    denominator = predictions.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return ((2 * intersection + eps) / (denominator + eps)).mean().item()

EPOCHS = 4
history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}'):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    scores = []
    with torch.no_grad():
        for images, masks in valid_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            scores.append(dice_score(model(images), masks))

    epoch_loss = running_loss / len(train_loader)
    epoch_dice = float(np.mean(scores))
    history.append((epoch_loss, epoch_dice))
    print(f'loss={epoch_loss:.4f} | validation Dice={epoch_dice:.4f}')

## validation

I inspect several validation predictions before generating the submission. The model usually finds the main object area, although the boundaries remain coarse.


In [ ]:
images, masks = next(iter(valid_loader))
model.eval()
with torch.no_grad():
    predictions = torch.sigmoid(model(images.to(DEVICE))).cpu() > 0.5

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(3):
    axes[i, 0].imshow(images[i].permute(1, 2, 0))
    axes[i, 0].set_title('Image')
    axes[i, 1].imshow(masks[i, 0], cmap='gray')
    axes[i, 1].set_title('Ground truth')
    axes[i, 2].imshow(predictions[i, 0], cmap='gray')
    axes[i, 2].set_title('Prediction')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()

## submission

The predicted 128 by 128 masks are expanded to 1280 by 1280 with nearest-neighbor interpolation and encoded in Fortran-order run-length encoding.


In [ ]:
def rle_encode(mask):
    pixels = (mask.flatten(order='F') > 0).astype(np.uint8)
    pixels = np.concatenate([[0], pixels, [0]])
    changes = np.where(pixels[1:] != pixels[:-1])[0] + 1
    changes[1::2] -= changes[::2]
    return ' '.join(map(str, changes))

test_dataset = PetGridDataset(test_df, BASE_DIR, with_masks=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

rles = []
model.eval()
with torch.no_grad():
    for images in tqdm(test_loader, desc='Inference'):
        probabilities = torch.sigmoid(model(images.to(DEVICE)))
        masks_128 = (probabilities > 0.5).cpu().numpy().astype(np.uint8)
        for mask in masks_128[:, 0]:
            mask_1280 = np.repeat(np.repeat(mask, 10, axis=0), 10, axis=1)
            rles.append(rle_encode(mask_1280))

submission = pd.DataFrame({'id': test_df['id'], 'mask_rle': rles})
submission.to_csv('submission.csv.gz', index=False, compression='gzip')
submission.head()

## result

The baseline reached a public Dice score of **0.3048**. The main improvement areas are skip connections, Dice loss, stronger augmentation, character-aware post-processing, and threshold tuning.
